# Lab 22: DPO Alignment for Vietnamese LLM — Complete Version

**Pipeline:** SFT-mini → DPO → Evaluation → GGUF Export

**Model:** Qwen2.5-3B (4-bit) | **GPU:** T4 16GB | **Est. time:** ~75 min

| Stage | Nội dung | Output |
|-------|----------|--------|
| 0 | Setup & Install | môi trường sẵn sàng |
| 1 | SFT-mini Training | `adapters/sft-mini/` |
| 2 | Preference Data Prep | `data/pref/train.parquet` |
| 3 | DPO Training | `adapters/dpo/` |
| 4 | Inference Comparison | `data/eval/side_by_side.jsonl` |
| 5 | Judge & Benchmark | `data/eval/judge_results.json` |
| 6 | Screenshots & Plots | `submission/screenshots/*.png` |
| 7 | GGUF Export | `gguf/lab22-dpo-Q4_K_M.gguf` |

## Stage 0: Setup & Install

In [ ]:
# Kiểm tra GPU trước
!nvidia-smi
!pip install -q unsloth vllm
!pip install -q --no-deps "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes
!pip install -q datasets transformers matplotlib pandas pyarrow


In [ ]:
import os, gc, json, time
from pathlib import Path
import torch
import matplotlib
matplotlib.use('Agg')  # headless safe
import matplotlib.pyplot as plt
import pandas as pd

# ── CONFIGURATION ─────────────────────────────────────────────
BASE_MODEL   = "unsloth/Qwen2.5-3B-bnb-4bit"
MAX_LEN      = 1024
SFT_SLICE    = 1000   # rows from Vietnamese Alpaca
PREF_SLICE   = 1000   # rows from UltraFeedback
EVAL_PROMPTS = 10     # prompts for side-by-side comparison

REPO_ROOT       = Path("/content/lab22")
SFT_PATH        = REPO_ROOT / "adapters" / "sft-mini"
DPO_PATH        = REPO_ROOT / "adapters" / "dpo"
GGUF_PATH       = REPO_ROOT / "gguf"
PREF_DATA_PATH  = REPO_ROOT / "data" / "pref"
EVAL_DATA_PATH  = REPO_ROOT / "data" / "eval"
SCREENSHOT_PATH = REPO_ROOT / "submission" / "screenshots"

for d in [SFT_PATH, DPO_PATH, GGUF_PATH, PREF_DATA_PATH,
          EVAL_DATA_PATH, SCREENSHOT_PATH]:
    d.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✓ Device     : {device}")
print(f"✓ CUDA mem   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"✓ Workspace  : {REPO_ROOT}")


## Stage 1: SFT-mini Training

In [ ]:
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# Load base model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name        = BASE_MODEL,
    max_seq_length    = MAX_LEN,
    load_in_4bit      = True,
    dtype             = None,   # auto-detect bf16/fp16
)
print(f"✓ Loaded base model: {BASE_MODEL}")


In [ ]:
# Attach LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r                     = 16,
    target_modules        = ["q_proj", "k_proj", "v_proj", "o_proj",
                              "gate_proj", "up_proj", "down_proj"],
    lora_alpha            = 16,
    lora_dropout          = 0,
    bias                  = "none",
    use_gradient_checkpointing = "unsloth",
    random_state          = 42,
)
print("✓ LoRA adapters attached")


In [ ]:
# Load & format Vietnamese Alpaca dataset
raw_sft = load_dataset(
    "vilm/vietnamese-alpaca-dataset-cleaned-52k",
    split=f"train[:{SFT_SLICE}]"
)

ALPACA_PROMPT = """Dưới đây là một nhiệm vụ mô tả công việc cần làm.
Hãy viết câu trả lời phù hợp.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

def format_alpaca(row):
    return {
        "text": ALPACA_PROMPT.format(
            instruction = row.get("instruction", ""),
            input       = row.get("input", ""),
            output      = row.get("output", ""),
        )
    }

dataset_sft = raw_sft.map(format_alpaca, remove_columns=raw_sft.column_names)
print(f"✓ SFT dataset: {len(dataset_sft)} samples")
print("Sample:")
print(dataset_sft[0]["text"][:300])


In [ ]:
USE_BF16 = torch.cuda.is_bf16_supported()

sft_trainer = SFTTrainer(
    model           = model,
    tokenizer       = tokenizer,
    train_dataset   = dataset_sft,
    dataset_text_field = "text",
    max_seq_length  = MAX_LEN,
    args = TrainingArguments(
        per_device_train_batch_size  = 2,
        gradient_accumulation_steps  = 4,
        max_steps                    = 60,
        learning_rate                = 2e-4,
        fp16                         = not USE_BF16,
        bf16                         = USE_BF16,
        logging_steps                = 10,
        warmup_steps                 = 5,
        output_dir                   = "outputs-sft",
        save_strategy                = "no",
        report_to                    = "none",
        seed                         = 42,
    ),
)

print("▶ Starting SFT training...")
t0 = time.time()
sft_trainer.train()
sft_time = time.time() - t0
print(f"✓ SFT done in {sft_time/60:.1f} min")

# Save SFT adapter + tokenizer
model.save_pretrained(str(SFT_PATH))
tokenizer.save_pretrained(str(SFT_PATH))
print(f"✓ SFT adapter saved → {SFT_PATH}")


In [ ]:
# Screenshot 01 — SFT Training Loss Curve
sft_logs = pd.DataFrame(sft_trainer.state.log_history)
sft_loss_logs = sft_logs.dropna(subset=["loss"])

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(sft_loss_logs["step"], sft_loss_logs["loss"],
        color="steelblue", marker="o", linewidth=2, label="Train Loss")
ax.set_title("Stage 1 — SFT-mini Training Loss", fontsize=14, fontweight="bold")
ax.set_xlabel("Step")
ax.set_ylabel("Loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(SCREENSHOT_PATH / "01-sft-training-log.png", dpi=150)
plt.show()
print("✓ Screenshot 01 saved")


In [ ]:
# Screenshot 02 — GPU Memory Usage
mem_alloc     = torch.cuda.memory_allocated(0) / 1e9
mem_reserved  = torch.cuda.memory_reserved(0)  / 1e9
mem_total     = torch.cuda.get_device_properties(0).total_memory / 1e9

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(
    ["Allocated", "Reserved", "Total (T4)"],
    [mem_alloc, mem_reserved, mem_total],
    color=["#e74c3c", "#e67e22", "#2ecc71"],
    edgecolor="white", linewidth=1.5
)
for bar, val in zip(bars, [mem_alloc, mem_reserved, mem_total]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f"{val:.2f} GB", ha="center", va="bottom", fontweight="bold")
ax.set_title("Stage 1 — GPU Memory After SFT", fontsize=14, fontweight="bold")
ax.set_ylabel("Memory (GB)")
ax.set_ylim(0, mem_total + 2)
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(SCREENSHOT_PATH / "02-gpu-memory.png", dpi=150)
plt.show()
print(f"✓ Screenshot 02 saved | Allocated: {mem_alloc:.2f} GB / {mem_total:.1f} GB")


## Stage 2: Preference Data Preparation

In [ ]:
from datasets import load_dataset

raw_pref = load_dataset(
    "argilla/ultrafeedback-binarized-preferences-cleaned",
    split=f"train[:{PREF_SLICE}]"
)
print(f"✓ Loaded {len(raw_pref)} preference rows")
print("Columns:", raw_pref.column_names)

def format_dpo(row):
    """
    UltraFeedback structure:
      - prompt: str
      - chosen:   list of {role, content} — last element is assistant response
      - rejected: list of {role, content} — last element is assistant response
    """
    chosen_msgs   = row.get("chosen", [])
    rejected_msgs = row.get("rejected", [])

    chosen_text   = chosen_msgs[-1]["content"]   if chosen_msgs   else ""
    rejected_text = rejected_msgs[-1]["content"] if rejected_msgs else ""

    return {
        "prompt":   row["prompt"],
        "chosen":   chosen_text,
        "rejected": rejected_text,
    }

dataset_pref = raw_pref.map(
    format_dpo,
    remove_columns=raw_pref.column_names
)

# Filter empty rows
dataset_pref = dataset_pref.filter(
    lambda x: len(x["prompt"]) > 10
              and len(x["chosen"]) > 10
              and len(x["rejected"]) > 10
)
print(f"✓ After filter: {len(dataset_pref)} samples")

# Save as parquet
dataset_pref.to_parquet(str(PREF_DATA_PATH / "train.parquet"))
print(f"✓ Saved → {PREF_DATA_PATH / 'train.parquet'}")

# Quick preview
sample = dataset_pref[0]
print(f"\nSample prompt   : {sample['prompt'][:100]}...")
print(f"Sample chosen   : {sample['chosen'][:100]}...")
print(f"Sample rejected : {sample['rejected'][:100]}...")


## Stage 3: DPO Training

In [ ]:
# Free memory before loading DPO model
del model, sft_trainer
gc.collect()
torch.cuda.empty_cache()
print(f"✓ Memory freed | Allocated: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")


In [ ]:
from unsloth import FastLanguageModel
from trl import DPOTrainer, DPOConfig

# Load SFT checkpoint as starting point for DPO
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = str(SFT_PATH),
    max_seq_length = MAX_LEN,
    load_in_4bit   = True,
    dtype          = None,
)
print("✓ SFT checkpoint loaded")

# Add fresh LoRA adapters for DPO
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ["q_proj", "v_proj"],
    lora_alpha     = 32,
    lora_dropout   = 0,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 42,
)
print("✓ DPO LoRA adapters attached")


In [ ]:
dpo_trainer = DPOTrainer(
    model     = model,
    ref_model = None,          # unsloth handles implicit ref via PEFT
    args = DPOConfig(
        output_dir                  = "outputs-dpo",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        max_steps                   = 40,
        learning_rate               = 5e-7,
        beta                        = 0.1,
        logging_steps               = 5,
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),
        warmup_steps                = 2,
        save_strategy               = "no",
        report_to                   = "none",
        seed                        = 42,
        max_prompt_length           = 512,
        max_length                  = MAX_LEN,
    ),
    train_dataset = dataset_pref,
    tokenizer     = tokenizer,
)

print("▶ Starting DPO training...")
t0 = time.time()
dpo_trainer.train()
dpo_time = time.time() - t0
print(f"✓ DPO done in {dpo_time/60:.1f} min")

# Save DPO adapter + tokenizer
model.save_pretrained(str(DPO_PATH))
tokenizer.save_pretrained(str(DPO_PATH))
print(f"✓ DPO adapter saved → {DPO_PATH}")


In [ ]:
# Save DPO metrics
dpo_logs = pd.DataFrame(dpo_trainer.state.log_history)
last_log = dpo_logs.iloc[-1].to_dict() if len(dpo_logs) > 0 else {}

metrics = {
    "model":           BASE_MODEL,
    "sft_steps":       60,
    "dpo_steps":       40,
    "dpo_beta":        0.1,
    "pref_samples":    len(dataset_pref),
    "final_loss":      float(last_log.get("loss", 0)),
    "final_rewards_chosen":   float(last_log.get("rewards/chosen",  0)),
    "final_rewards_rejected": float(last_log.get("rewards/rejected", 0)),
    "final_reward_margin":    float(last_log.get("rewards/margins",  0)),
    "end_reward_gap":         float(last_log.get("rewards/margins",  0)),
    "sft_train_time_min":     round(sft_time / 60, 2),
    "dpo_train_time_min":     round(dpo_time / 60, 2),
}

with open(DPO_PATH / "dpo_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("✓ DPO metrics saved:")
print(json.dumps(metrics, indent=2))


In [ ]:
# Screenshot 03 — DPO Reward Curves
reward_logs = dpo_logs.dropna(subset=["rewards/chosen"]).copy()

if len(reward_logs) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Reward curves
    ax = axes[0]
    ax.plot(reward_logs["step"], reward_logs["rewards/chosen"],
            label="Chosen", color="green", marker="o", linewidth=2)
    ax.plot(reward_logs["step"], reward_logs["rewards/rejected"],
            label="Rejected", color="red", marker="x", linewidth=2)
    ax.set_title("DPO Reward Curves", fontsize=13, fontweight="bold")
    ax.set_xlabel("Step")
    ax.set_ylabel("Reward")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Reward margin
    ax2 = axes[1]
    if "rewards/margins" in reward_logs.columns:
        ax2.plot(reward_logs["step"], reward_logs["rewards/margins"],
                 color="purple", marker="s", linewidth=2)
        ax2.set_title("DPO Reward Margin (Chosen − Rejected)",
                      fontsize=13, fontweight="bold")
        ax2.set_xlabel("Step")
        ax2.set_ylabel("Margin")
        ax2.axhline(0, color="black", linestyle="--", alpha=0.5)
        ax2.grid(True, alpha=0.3)

    plt.suptitle("Stage 3 — DPO Training Metrics", fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.savefig(SCREENSHOT_PATH / "03-dpo-reward-curves.png", dpi=150)
    plt.show()
    print("✓ Screenshot 03 saved")
else:
    print("⚠ No reward logs found — skipping reward plot")
    # Still create a placeholder screenshot
    loss_logs = dpo_logs.dropna(subset=["loss"])
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(loss_logs["step"], loss_logs["loss"],
            color="steelblue", marker="o", linewidth=2)
    ax.set_title("Stage 3 — DPO Training Loss", fontsize=14, fontweight="bold")
    ax.set_xlabel("Step")
    ax.set_ylabel("Loss")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(SCREENSHOT_PATH / "03-dpo-reward-curves.png", dpi=150)
    plt.show()


## Stage 4: Inference Comparison (Base vs SFT vs DPO)

In [ ]:
# Define evaluation prompts (mix of Vietnamese and English)
EVAL_PROMPTS_LIST = [
    "Giải thích khái niệm học máy cho người mới bắt đầu.",
    "Viết một đoạn văn ngắn về lợi ích của việc đọc sách.",
    "Làm thế nào để cải thiện kỹ năng giao tiếp?",
    "Phân tích nguyên nhân của biến đổi khí hậu.",
    "Hãy kể một câu chuyện ngắn về tình bạn.",
    "Explain the difference between supervised and unsupervised learning.",
    "What are the key principles of effective leadership?",
    "Describe the water cycle in simple terms.",
    "Viết code Python để tính giai thừa của một số.",
    "Những kỹ năng nào quan trọng nhất trong thế kỷ 21?",
]

print(f"✓ {len(EVAL_PROMPTS_LIST)} evaluation prompts ready")


In [ ]:
from unsloth import FastLanguageModel

def run_inference(model_path_or_name, prompts, max_new_tokens=200):
    """Load a model and generate responses for all prompts."""
    m, tok = FastLanguageModel.from_pretrained(
        model_name     = str(model_path_or_name),
        max_seq_length = MAX_LEN,
        load_in_4bit   = True,
        dtype          = None,
    )
    FastLanguageModel.for_inference(m)

    results = []
    for prompt in prompts:
        formatted = ALPACA_PROMPT.format(
            instruction = prompt,
            input       = "",
            output      = "",          # empty → model will complete
        )
        inputs = tok(formatted, return_tensors="pt").to("cuda")
        with torch.no_grad():
            out = m.generate(
                **inputs,
                max_new_tokens = max_new_tokens,
                do_sample      = True,
                temperature    = 0.7,
                top_p          = 0.9,
                pad_token_id   = tok.eos_token_id,
            )
        # Decode only the generated part
        generated = tok.decode(
            out[0][inputs["input_ids"].shape[-1]:],
            skip_special_tokens=True
        ).strip()
        results.append(generated)

    del m
    gc.collect()
    torch.cuda.empty_cache()
    return results

ALPACA_PROMPT = """Dưới đây là một nhiệm vụ mô tả công việc cần làm.
Hãy viết câu trả lời phù hợp.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

print("✓ Inference function defined")


In [ ]:
print("▶ Running inference on SFT model...")
sft_responses = run_inference(SFT_PATH, EVAL_PROMPTS_LIST)
print(f"✓ SFT responses: {len(sft_responses)}")

print("▶ Running inference on DPO model...")
dpo_responses = run_inference(DPO_PATH, EVAL_PROMPTS_LIST)
print(f"✓ DPO responses: {len(dpo_responses)}")

# Reload DPO model as current model (needed for GGUF stage)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = str(DPO_PATH),
    max_seq_length = MAX_LEN,
    load_in_4bit   = True,
    dtype          = None,
)
print("✓ DPO model reloaded as current model")


In [ ]:
# Save side-by-side comparison
side_by_side = []
for i, prompt in enumerate(EVAL_PROMPTS_LIST):
    side_by_side.append({
        "id":           i + 1,
        "prompt":       prompt,
        "sft_response": sft_responses[i],
        "dpo_response": dpo_responses[i],
    })

sbs_path = EVAL_DATA_PATH / "side_by_side.jsonl"
with open(sbs_path, "w", encoding="utf-8") as f:
    for row in side_by_side:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"✓ Saved → {sbs_path}")

# Print one example
ex = side_by_side[0]
print(f"\nPrompt: {ex['prompt']}")
print(f"\nSFT: {ex['sft_response'][:200]}")
print(f"\nDPO: {ex['dpo_response'][:200]}")


In [ ]:
# Screenshot 04 — Inference Comparison
fig, axes = plt.subplots(len(side_by_side[:3]), 1, figsize=(14, 12))
if len(side_by_side[:3]) == 1:
    axes = [axes]

for ax, row in zip(axes, side_by_side[:3]):
    ax.axis("off")
    text = (
        f"Q: {row['prompt']}\n\n"
        f"SFT: {row['sft_response'][:200]}\n\n"
        f"DPO: {row['dpo_response'][:200]}"
    )
    ax.text(0, 1, text, transform=ax.transAxes, fontsize=9,
            verticalalignment="top", wrap=True,
            bbox=dict(boxstyle="round", facecolor="#f0f8ff", alpha=0.8))

fig.suptitle("Stage 4 — Inference: SFT vs DPO Comparison",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(SCREENSHOT_PATH / "04-side-by-side-table.png", dpi=150)
plt.show()
print("✓ Screenshot 04 saved")


## Stage 5: Judge & Benchmark

In [ ]:
import re

# Heuristic judge: compare response quality using simple metrics
# (In production, you'd use an LLM-as-judge; here we use length + diversity proxies)
def heuristic_judge(sft_resp, dpo_resp):
    """Returns 'dpo', 'sft', or 'tie' based on simple heuristics."""
    sft_len   = len(sft_resp.split())
    dpo_len   = len(dpo_resp.split())
    sft_words = len(set(sft_resp.lower().split()))
    dpo_words = len(set(dpo_resp.lower().split()))

    # Score: weighted mix of length (40%) and vocabulary diversity (60%)
    sft_score = 0.4 * sft_len + 0.6 * sft_words
    dpo_score = 0.4 * dpo_len + 0.6 * dpo_words

    diff = (dpo_score - sft_score) / max(sft_score, 1)
    if   diff >  0.05: return "dpo"
    elif diff < -0.05: return "sft"
    else:              return "tie"

judge_results = []
win_counts = {"dpo": 0, "sft": 0, "tie": 0}

for row in side_by_side:
    winner = heuristic_judge(row["sft_response"], row["dpo_response"])
    win_counts[winner] += 1
    judge_results.append({
        "id":         row["id"],
        "prompt":     row["prompt"],
        "winner":     winner,
        "sft_words":  len(row["sft_response"].split()),
        "dpo_words":  len(row["dpo_response"].split()),
        "sft_vocab":  len(set(row["sft_response"].lower().split())),
        "dpo_vocab":  len(set(row["dpo_response"].lower().split())),
    })

with open(EVAL_DATA_PATH / "judge_results.json", "w", encoding="utf-8") as f:
    json.dump({"summary": win_counts, "results": judge_results}, f,
              indent=2, ensure_ascii=False)

print(f"✓ Judge results: {win_counts}")
dpo_win_rate = win_counts['dpo'] / len(judge_results) * 100
print(f"  DPO win rate: {dpo_win_rate:.0f}%")


In [ ]:
# Compute response quality metrics
def avg_metric(results, key):
    return round(sum(r[key] for r in results) / len(results), 2)

benchmark = {
    "model_base":       BASE_MODEL,
    "eval_prompts":     len(judge_results),
    "sft": {
        "avg_words":    avg_metric(judge_results, "sft_words"),
        "avg_vocab":    avg_metric(judge_results, "sft_vocab"),
        "win_count":    win_counts["sft"],
    },
    "dpo": {
        "avg_words":    avg_metric(judge_results, "dpo_words"),
        "avg_vocab":    avg_metric(judge_results, "dpo_vocab"),
        "win_count":    win_counts["dpo"],
        "win_rate_pct": round(dpo_win_rate, 1),
    },
    "ties":             win_counts["tie"],
    "dpo_metrics":      metrics,
}

with open(EVAL_DATA_PATH / "benchmark_results.json", "w", encoding="utf-8") as f:
    json.dump(benchmark, f, indent=2, ensure_ascii=False)

print("✓ Benchmark results saved:")
print(json.dumps(benchmark, indent=2))


In [ ]:
# Screenshot 05 — Benchmark Results
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Win rate pie
ax = axes[0]
labels = [f"DPO ({win_counts['dpo']})",
          f"SFT ({win_counts['sft']})",
          f"Tie ({win_counts['tie']})"]
ax.pie([win_counts["dpo"], win_counts["sft"], win_counts["tie"]],
       labels=labels, colors=["#2ecc71", "#e74c3c", "#95a5a6"],
       autopct="%1.0f%%", startangle=90)
ax.set_title("Judge Win Rate", fontweight="bold")

# Avg words
ax2 = axes[1]
ax2.bar(["SFT", "DPO"],
        [benchmark["sft"]["avg_words"], benchmark["dpo"]["avg_words"]],
        color=["#e74c3c", "#2ecc71"], edgecolor="white")
ax2.set_title("Avg Response Length (words)", fontweight="bold")
ax2.set_ylabel("Words")
ax2.grid(True, axis="y", alpha=0.3)

# Avg vocab diversity
ax3 = axes[2]
ax3.bar(["SFT", "DPO"],
        [benchmark["sft"]["avg_vocab"], benchmark["dpo"]["avg_vocab"]],
        color=["#e74c3c", "#2ecc71"], edgecolor="white")
ax3.set_title("Avg Vocabulary Diversity", fontweight="bold")
ax3.set_ylabel("Unique words")
ax3.grid(True, axis="y", alpha=0.3)

fig.suptitle("Stage 5 — SFT vs DPO Benchmark", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(SCREENSHOT_PATH / "07-benchmark-comparison.png", dpi=150)
plt.show()
print("✓ Screenshot 05 saved")


## Stage 6: Summary Screenshots

In [ ]:
# Screenshot 06 — Pre-GGUF model info
fig, ax = plt.subplots(figsize=(12, 6))
ax.axis("off")

info_text = (
    f"Lab 22 — DPO Alignment Pipeline\n"
    f"{'='*60}\n"
    f"Base Model    : {BASE_MODEL}\n"
    f"SFT Steps     : 60  |  LR: 2e-4  |  Data: {SFT_SLICE} samples\n"
    f"DPO Steps     : 40  |  LR: 5e-7  |  Beta: 0.1\n"
    f"Pref Data     : {len(dataset_pref)} samples (UltraFeedback)\n"
    f"SFT Train Time: {sft_time/60:.1f} min\n"
    f"DPO Train Time: {dpo_time/60:.1f} min\n"
    f"{'='*60}\n"
    f"Output Paths:\n"
    f"  adapters/sft-mini/  : LoRA adapter (SFT)\n"
    f"  adapters/dpo/       : LoRA adapter (DPO)\n"
    f"  data/pref/          : train.parquet\n"
    f"  data/eval/          : judge & benchmark JSON\n"
    f"  gguf/               : Q4_K_M export (next stage)\n"
    f"{'='*60}\n"
    f"DPO Win Rate  : {dpo_win_rate:.0f}% (heuristic judge)"
)

ax.text(0.05, 0.95, info_text, transform=ax.transAxes, fontsize=11,
        verticalalignment="top", fontfamily="monospace",
        bbox=dict(boxstyle="round", facecolor="#1e1e2e", alpha=0.9),
        color="#cdd6f4")

fig.suptitle("Stage 6 — Pipeline Summary (Pre-GGUF Export)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(SCREENSHOT_PATH / "06-pipeline-summary.png", dpi=150)
plt.show()
print("✓ Screenshot 06 saved")


## Stage 7: GGUF Export

In [ ]:
print("▶ Exporting to GGUF Q4_K_M...")
t0 = time.time()

model.save_pretrained_gguf(
    str(GGUF_PATH),
    tokenizer,
    quantization_method = "q4_k_m",
)

gguf_time = time.time() - t0

# Find the actual .gguf file
gguf_files = list(GGUF_PATH.glob("*.gguf"))
print(f"✓ GGUF export done in {gguf_time/60:.1f} min")
for f in gguf_files:
    size_gb = f.stat().st_size / 1e9
    print(f"  {f.name}: {size_gb:.2f} GB")


In [ ]:
# Screenshot 07 — Final Summary
gguf_files = list(GGUF_PATH.glob("*.gguf"))
gguf_info  = [(f.name, f.stat().st_size/1e9) for f in gguf_files]

fig, ax = plt.subplots(figsize=(12, 6))
ax.axis("off")

gguf_lines = "\n".join(
    f"  {name}: {size:.2f} GB" for name, size in gguf_info
) or "  (checking...)"

summary = (
    f"Lab 22 — COMPLETE ✓\n"
    f"{'='*60}\n"
    f"GGUF Files:\n{gguf_lines}\n"
    f"Export Time   : {gguf_time/60:.1f} min\n"
    f"{'='*60}\n"
    f"Submission Checklist:\n"
    f"  ✓ adapters/sft-mini/adapter_config.json\n"
    f"  ✓ adapters/dpo/adapter_config.json\n"
    f"  ✓ adapters/dpo/dpo_metrics.json\n"
    f"  ✓ data/pref/train.parquet\n"
    f"  ✓ data/eval/side_by_side.jsonl\n"
    f"  ✓ data/eval/judge_results.json\n"
    f"  ✓ data/eval/benchmark_results.json\n"
    f"  ✓ submission/screenshots/ (7 images)\n"
    f"  ✓ gguf/lab22-dpo-Q4_K_M.gguf\n"
    f"  ✗ submission/REFLECTION.md (viết thủ công)"
)

ax.text(0.05, 0.95, summary, transform=ax.transAxes, fontsize=11,
        verticalalignment="top", fontfamily="monospace",
        bbox=dict(boxstyle="round", facecolor="#1a2e1a", alpha=0.9),
        color="#a8d8a8")

fig.suptitle("Stage 7 — GGUF Export Complete", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(SCREENSHOT_PATH / "07-benchmark-comparison.png", dpi=150)
plt.show()
print("✓ Screenshot 07 saved")


## Final Verification

In [ ]:
print("=" * 60)
print("FINAL SUBMISSION CHECKLIST")
print("=" * 60)

required_files = [
    SFT_PATH / "adapter_config.json",
    DPO_PATH / "adapter_config.json",
    DPO_PATH / "dpo_metrics.json",
    PREF_DATA_PATH  / "train.parquet",
    EVAL_DATA_PATH  / "side_by_side.jsonl",
    EVAL_DATA_PATH  / "judge_results.json",
    EVAL_DATA_PATH  / "benchmark_results.json",
    SCREENSHOT_PATH / "01-sft-training-log.png",
    SCREENSHOT_PATH / "02-gpu-memory.png",
    SCREENSHOT_PATH / "03-dpo-reward-curves.png",
    SCREENSHOT_PATH / "04-side-by-side-table.png",
    SCREENSHOT_PATH / "07-benchmark-comparison.png",
    SCREENSHOT_PATH / "06-pipeline-summary.png",
    SCREENSHOT_PATH / "07-benchmark-comparison.png",
]

all_ok = True
for fp in required_files:
    exists = fp.exists()
    status = "✓" if exists else "✗ MISSING"
    if not exists:
        all_ok = False
    size = f"({fp.stat().st_size/1e3:.0f} KB)" if exists else ""
    print(f"  {status}  {fp.relative_to(REPO_ROOT)}  {size}")

# GGUF
gguf_files = list(GGUF_PATH.glob("*.gguf"))
if gguf_files:
    for gf in gguf_files:
        print(f"  ✓  gguf/{gf.name} ({gf.stat().st_size/1e9:.2f} GB)")
else:
    print("  ✗ MISSING  gguf/*.gguf")
    all_ok = False

print()
if all_ok:
    print("✅ ALL REQUIRED FILES PRESENT — ready to commit!")
else:
    print("⚠  Some files missing — re-run the relevant cells above.")

print()
print("⚠  Nhớ tự viết submission/REFLECTION.md (>150 từ/phần) trước khi nộp.")
